# Clustering Models — Land Use Segmentation

K-Means, DBSCAN, and hierarchical clustering with PCA reduction.
Silhouette analysis for optimal cluster count.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

## Modeling Steps

1. Scale features and apply PCA for dimensionality reduction
2. Run K-Means with silhouette analysis for k=3..10
3. Run DBSCAN with epsilon tuning
4. Run hierarchical agglomerative clustering
5. Compare cluster quality metrics

In [ ]:
# Load and scale features
df = pd.read_parquet('../data/processed/pixel_features.parquet')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.values)

# PCA reduction
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
print(f'Explained variance: {pca.explained_variance_ratio_.cumsum()[-1]:.3f}')

In [ ]:
# Silhouette analysis for K-Means
k_range = range(3, 11)
silhouettes = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels, sample_size=5000)
    silhouettes.append(sil)
    print(f'k={k}: silhouette={sil:.4f}')

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'\nBest k={best_k} (silhouette={max(silhouettes):.4f})')

# Final K-Means with best k
km_final = KMeans(n_clusters=7, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(X_pca)
df.to_parquet('../data/processed/clustered_pixels.parquet', index=False)